# Score `patch_s2` on the GIFT-Eval test split (once)

`patch_s2` (10,083,392 parameters) was chosen on the validation split by the rule in
`PREREGISTRATION.md` section 4. Its checkpoint stayed on Google Drive, so it has no
test score yet. This notebook gives it the pre-registered one.

**What the pre-registration fixes, and this notebook follows**

* Test split: the 97 official configurations, at the pinned revision (section 2).
* Metrics: GluonTS MASE and CRPS over Seasonal Naive, geometric mean (section 3).
* Inference: context up to 2,048 points, patch roll-out by feeding back medians (section 5).
* Decision rule: CRPS against upstream AutoARIMA, 95% paired bootstrap over
  configurations; better, worse or no detectable difference (section 5).
* Scored once (section 6). `tsfm eval` refuses a second test score, and refuses any
  model test score unless the `prereg-v1` tag is in the commit's history.

**What this run is not.** It is one seed. `DEVIATIONS.md` (2026-10-15) already records
that seeds 1 and 2 were not trained, so the interval covers configurations, not seeds.
It is not a second look: the 2026-10-15 attempt disconnected partway and its partial
outputs were never inspected. This notebook writes to a new folder and never reads them.
`PREREGISTRATION_CPU.md` does not authorise or forbid this run; the CPU study is a
separate model with its own test score.

**Steps.** Set `CHECKPOINT_PATH` below, then Runtime -> Run all on a T4 GPU runtime.
Cell 5 checks the checkpoint and stops before the test split if it is not `patch_s2`.
After a disconnect, Run all again: finished configurations are kept on Drive. The last
cell prints the result files with SHA-256 for `scripts/harvest_colab_output.py`.

In [ ]:
# Parameters
# ============================================================================
# CHECKPOINT_PATH: the only value to set. The patch_s2 run folder on Drive.
# ckpt.pt (training checkpoint) or model.pt (inference weights) both work: the
# script checks the run config and the 10,083,392 parameters, then scores the
# file whose SHA-256 matches the validation run.
CHECKPOINT_PATH = "/content/drive/MyDrive/tsfm/runs/patch_s2/ckpt.pt"
# ============================================================================
REPO = "shmorsi/tsfm-from-scratch"
COMMIT = "main"  # or a full commit SHA
WORK = "/content/drive/MyDrive/tsfm/score_patch_s2"  # resumable outputs

In [ ]:
# 1. GPU check
import subprocess

import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive (checkpoint in, resumable results out)
import os

from google.colab import drive

drive.mount("/content/drive")
assert os.path.isfile(CHECKPOINT_PATH), f"CHECKPOINT_PATH not found: {CHECKPOINT_PATH}"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# 3. Clone the repo at COMMIT, with tags (tsfm eval checks for prereg-v1)
import os
import subprocess

if not os.path.exists("/content/tsfm"):
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/content/tsfm"], check=True)
subprocess.run(["git", "-C", "/content/tsfm", "fetch", "-q", "--force", "--tags", "origin"], check=True)
remote = f"origin/{COMMIT}"
known = subprocess.run(["git", "-C", "/content/tsfm", "rev-parse", "-q", "--verify", remote], capture_output=True)
ref = remote if known.returncode == 0 else COMMIT
subprocess.run(["git", "-C", "/content/tsfm", "checkout", "-q", "--force", "--detach", ref], check=True)
print(subprocess.run(["git", "-C", "/content/tsfm", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 4. Install (uv, locked dependencies, CUDA 12.6 PyTorch build)
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["TSFM_HOME"] = "/content/tsfm_home"
%cd /content/tsfm
!uv sync --frozen --extra cu126 -q
!uv run python -c 'import torch; print(torch.__version__, torch.cuda.is_available())'

In [ ]:
# 5. Check the checkpoint is patch_s2 before anything touches the test split
!uv run python scripts/score_patch_s2.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --dry-run

In [ ]:
# 6. Score once: GIFT-Eval download (SHA-256 checked), tsfm eval --split test,
#    tsfm report, tsfm headline, tsfm docs build. Re-run after a disconnect.
!uv run python scripts/score_patch_s2.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --device cuda

In [ ]:
# 7. Result files with SHA-256. Copy this output into a text file, then, in the repo:
#    uv run python scripts/harvest_colab_output.py <that file> && uv run tsfm docs build
print(open(f"{WORK}/harvest_patch_s2.txt").read())
!git -C /content/tsfm status --short